# Centriole geometry perturbation analysis

Uses `centriole.py` (the geometric model) and `perturbation.py` (sweep/comparison helpers) to explore how changes to the underlying parameters -- standing in for biological perturbations -- change the predicted centriole cross-section.

Two worked examples below, matching the motivating cases:
1. **A SAS-6 mutation that drops symmetry from 9-fold to 8-fold** (or lower) -- a discrete, qualitative perturbation. Compared with `plot_grid`.
2. **A longer SAS-6 coiled-coil** (increasing `r`) -- a continuous, quantitative perturbation. Compared with `sweep_param`.

Swap in your own parameter sets below to test other perturbations.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from centriole import centriole
from perturbation import sweep_param, plot_grid

%matplotlib inline

## Baseline (wild-type) geometry

A reference configuration to perturb against. Adjust to match your organism/structure of interest.

In [ ]:
WT = dict(SYM=9, MTn=3, LK=25, r=77, L=50, MT_RADIUS=10, GAMMA=60)

fig, ax = plt.subplots(figsize=(5, 5))
result = centriole(**WT, ax=ax)
result

## Example 1: symmetry-breaking mutation (9-fold → 8-fold, 7-fold)

A discrete/qualitative perturbation — `plot_grid` renders each configuration as its own panel, sharing every other parameter with `WT`, and returns a table of the numeric outcomes (`success`, `overlap`, `b_deg`, achieved `LKm`) alongside the figure.

In [ ]:
fig, table = plot_grid(
    {
        "WT (9-fold)": {"SYM": 9},
        "8-fold mutant": {"SYM": 8},
        "7-fold mutant": {"SYM": 7},
    },
    **{k: v for k, v in WT.items() if k != "SYM"},
)
table

## Example 2: longer SAS-6 coiled-coil (sweep `r`)

A continuous/quantitative perturbation — `sweep_param` holds everything else fixed at `WT` and varies `r`, returning a tidy DataFrame. Useful for asking things like "at what SAS-6 length does the model stop finding a non-overlapping solution?"

In [ ]:
df = sweep_param("r", range(20, 100, 2), **{k: v for k, v in WT.items() if k != "r"})
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df["r"], df["b_deg"], marker="o")
axes[0].set_xlabel("SAS-6 length r")
axes[0].set_ylabel("tilt angle b (deg)")

axes[1].plot(df["r"], df["LKm"], marker="o")
axes[1].axhline(WT["LK"], color="gray", linestyle="--", label="target LK")
axes[1].set_xlabel("SAS-6 length r")
axes[1].set_ylabel("achieved LK")
axes[1].legend()

axes[2].scatter(df["r"], df["success"].astype(int), label="success", marker="o")
axes[2].scatter(df["r"], df["overlap"].astype(int) - 0.05, label="overlap", marker="x")
axes[2].set_xlabel("SAS-6 length r")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["False", "True"])
axes[2].legend()

fig.tight_layout()

## Try your own perturbation

Copy a cell above and change the varied parameter/values -- e.g. `GAMMA` (linker anchoring angle), `MT_RADIUS`, or `MTn` (singlet/doublet/triplet) -- to test other hypotheses.

In [ ]:
fig, table = plot_grid(
    {
        "WT (9-fold)": {"SYM": 9},
        "8-fold mutant": {"SYM": 8},
        "7-fold mutant": {"SYM": 7},
    },
    **{k: v for k, v in WT.items() if k != "SYM"},
)
table
df = sweep_param("r", range(20, 200, 2), **{k: v for k, v in WT.items() if k != "r"})
df.head()

## Part 2: anatomically detailed model (`centriole_v2`)

`centriole_v2` decomposes the lumped SAS-6 spoke length `r` into
separately named substructures (CID, cartwheel hub+spokes, pinhead,
triplet base, MT triplet), matched to a reference cryo-ET cross-section.
Same underlying search/overlap algorithm as `centriole`, same
`sweep_param`/`plot_grid` tooling -- just pass `model=centriole_v2`.

**Caveat:** default segment lengths are eyeballed proportions from a
reference image, not precise measurements -- calibrate before drawing
real conclusions. See the README for the full parameter mapping.

In [ ]:
from centriole_v2 import centriole_v2

WT_V2 = dict(SYM=9, LK=25, MTn=3)

fig, ax = plt.subplots(figsize=(5, 5))
result = centriole_v2(**WT_V2, ax=ax)
result

### Symmetry-breaking mutation, v2 model

In [ ]:
fig, table = plot_grid(
    {"WT (9-fold)": {"SYM": 9}, "8-fold mutant": {"SYM": 8}, "7-fold mutant": {"SYM": 7}},
    model=centriole_v2,
    LK=25,
)
table

### Which segment drives a longer SAS-6 reach? Compare spoke vs. pinhead vs. base

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, param in zip(axes, ["SPOKE_LENGTH", "PINHEAD_LENGTH", "BASE_LENGTH"]):
    df = sweep_param(param, range(5, 45, 5), model=centriole_v2, SYM=9, LK=25)
    ax.plot(df[param], df["b_deg"], marker="o", label="tilt angle b")
    ax2 = ax.twinx()
    ax2.plot(df[param], df["LKm"], marker="x", color="darkorange", label="achieved LK")
    ax.set_xlabel(param)
    ax.set_ylabel("tilt angle b (deg)")
    ax2.set_ylabel("achieved LK")
    ax.set_title(param)

fig.tight_layout()

**Note on the three curves above:** they have the same shape. That is
expected, not a bug -- `Rc = HUB_RADIUS + SPOKE_LENGTH + PINHEAD_LENGTH +
BASE_LENGTH` is purely additive, so the angle search only ever sees the
*total* radial reach, not which segment contributed it. Perturbing any
one segment by the same amount has an identical effect on `b_deg`/`LKm`.
The decomposition is useful for setting segment-specific defaults and for
visualizing *where* a perturbation acts anatomically, but this version of
the model does not yet distinguish segments by anything other than
length (no stiffness, angle, or attachment-chemistry differences) --
extending it that way would be the natural next step if a hypothesized
perturbation isn't just "this segment got longer/shorter."